<a href="https://colab.research.google.com/github/simoesya/ProjetoFinal/blob/main/Projeto_RAG_TriviaQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Environment and output paths

In [1]:
%pip -q install rank-bm25 'transformers>=5,<6' accelerate

import gc
import json
import re
import subprocess
import sys
import tarfile
import urllib.request
from collections import Counter
from pathlib import Path

import torch
from google.colab import drive
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Ativa a GPU no ambiente de execução do Colab.")

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Projeto_RAG_TriviaQA/reproducao")
destino = PROJECT_DIR / "results"
destino.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("/content/triviaqa_local")

def ler_json(caminho):
    return json.loads(caminho.read_text(encoding="utf-8"))

def guardar_json(caminho, dados):
    temporario = caminho.with_suffix(".tmp")
    temporario.write_text(
        json.dumps(dados, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    temporario.replace(caminho)


Mounted at /content/drive


## 2. Dataset and official evaluation functions

In [2]:
arquivo_local = Path("/content/triviaqa-rc.tar.gz")
marcador = DATA_DIR / ".wikipedia_extraida"
ficheiro_treino = DATA_DIR / "qa/wikipedia-train.json"
ficheiro_teste = DATA_DIR / "qa/verified-wikipedia-dev.json"

if not all(p.is_file() for p in (marcador, ficheiro_treino, ficheiro_teste)):
    if not arquivo_local.is_file():
        print("A descarregar TriviaQA...", flush=True)
        temporario = arquivo_local.with_suffix(".gz.part")
        urllib.request.urlretrieve(
            "https://nlp.cs.washington.edu/triviaqa/data/triviaqa-rc.tar.gz",
            temporario,
        )
        temporario.replace(arquivo_local)

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("A extrair os ficheiros Wikipedia localmente...", flush=True)
    with tarfile.open(arquivo_local, "r:gz") as tar:
        for membro in tar:
            nome = membro.name.removeprefix("./")
            if membro.isfile() and (
                nome in ("qa/wikipedia-train.json", "qa/verified-wikipedia-dev.json")
                or nome.startswith("evidence/wikipedia/")
            ):
                tar.extract(membro, path=DATA_DIR, filter="data")
    if not all(p.is_file() for p in (ficheiro_treino, ficheiro_teste)):
        raise RuntimeError("A extração não encontrou os JSON esperados.")
    marcador.touch()

perguntas = ler_json(ficheiro_treino)["Data"]
validacao, treino = perguntas[:7900], perguntas[7900:]
teste = ler_json(ficheiro_teste)["Data"]
assert (len(treino), len(validacao), len(teste)) == (53988, 7900, 318)
validacao_200 = validacao[:200]

pasta_divisoes = PROJECT_DIR / "data" / "splits"
pasta_divisoes.mkdir(parents=True, exist_ok=True)
conjuntos = {"treino": treino, "validacao": validacao, "teste": teste}
ids = [{p["QuestionId"] for p in conjunto} for conjunto in conjuntos.values()]
assert sum(map(len, ids)) == len(set.union(*ids))
for nome, conjunto in conjuntos.items():
    guardar_json(pasta_divisoes / f"{nome}.json", conjunto)
    print(f"{nome}: {len(conjunto)} perguntas")


A descarregar TriviaQA...
A extrair os ficheiros Wikipedia localmente...
treino: 53988 perguntas
validacao: 7900 perguntas
teste: 318 perguntas


In [3]:
REPO_DIR = Path("/content/triviaqa_official")
if not (REPO_DIR / "evaluation/triviaqa_evaluation.py").is_file():
    subprocess.run(
        ["git", "clone", "https://github.com/mandarjoshi90/triviaqa.git", str(REPO_DIR)],
        check=True,
    )
sys.path.insert(0, str(REPO_DIR))
from evaluation import triviaqa_evaluation as scorer

commit_avaliador = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("Commit do avaliador:", commit_avaliador)

def aliases_da_resposta(resposta):
    return {
        scorer.normalize_answer(alias)
        for alias in [resposta["Value"], *resposta["NormalizedAliases"]]
    } - {""}

def contem_resposta(texto, aliases):
    normalizado = f" {scorer.normalize_answer(texto)} "
    return any(f" {alias} " in normalizado for alias in aliases)


Commit do avaliador: ca43b5820b107f3970cf4b7d67f7db7a98117b79


## 3. Passage retrieval

Split documents into 200-word passages with 50 words of overlap. Evaluate TF-IDF and BM25 on the same questions using alias-based Recall@1, @5 and @10.

In [4]:
def dividir_em_excertos(texto, tamanho=200, sobreposicao=50):
    if not 0 <= sobreposicao < tamanho:
        raise ValueError("A sobreposição deve estar entre 0 e tamanho - 1.")

    palavras = texto.split()
    excertos = []
    passo = tamanho - sobreposicao

    for inicio in range(0, len(palavras), passo):
        excertos.append(" ".join(palavras[inicio:inicio + tamanho]))

        if inicio + tamanho >= len(palavras):
            break

    return excertos


def pesquisar_excertos(pergunta, k=10):
    candidatos = []
    ficheiros = dict.fromkeys(doc["Filename"] for doc in pergunta["EntityPages"])

    for ficheiro in ficheiros:
        caminho = DATA_DIR / "evidence" / "wikipedia" / ficheiro
        texto = caminho.read_text(encoding="utf-8")

        for numero, excerto in enumerate(dividir_em_excertos(texto)):
            candidatos.append({
                "ficheiro": ficheiro,
                "numero": numero,
                "texto": excerto
            })

    if not candidatos:
        return []

    vetorizador = TfidfVectorizer(stop_words="english")
    matriz = vetorizador.fit_transform([item["texto"] for item in candidatos])
    consulta = vetorizador.transform([pergunta["Question"]])

    scores = cosine_similarity(consulta, matriz).ravel()
    indices = scores.argsort()[::-1][:k]

    return [
        {**candidatos[int(i)], "score": float(scores[i])}
        for i in indices
    ]



def tokenizar_bm25(texto):
    return re.findall(r"\w+", texto.lower())

def pesquisar_bm25(pergunta, k=10):
    candidatos = []
    tokens_candidatos = []
    ficheiros = dict.fromkeys(
        doc["Filename"] for doc in pergunta["EntityPages"]
    )

    for ficheiro in ficheiros:
        caminho = DATA_DIR / "evidence" / "wikipedia" / ficheiro
        texto = caminho.read_text(encoding="utf-8")

        for numero, excerto in enumerate(dividir_em_excertos(texto)):
            tokens = tokenizar_bm25(excerto)
            if tokens:
                candidatos.append({
                    "ficheiro": ficheiro,
                    "numero": numero,
                    "texto": excerto
                })
                tokens_candidatos.append(tokens)

    if not candidatos:
        return []

    indice = BM25Okapi(tokens_candidatos, k1=1.5, b=0.75, epsilon=0.25)
    scores = indice.get_scores(tokenizar_bm25(pergunta["Question"]))
    melhores = scores.argsort()[::-1][:k]

    return [
        {**candidatos[int(i)], "score": float(scores[i])}
        for i in melhores
    ]


In [5]:
metodos_pesquisa = {"tfidf": pesquisar_excertos, "bm25": pesquisar_bm25}

def avaliar_recuperacao(perguntas, conjunto):
    rankings = {}
    for nome, pesquisar in metodos_pesquisa.items():
        registos = []
        for pergunta in tqdm(perguntas, desc=f"{conjunto}: {nome}"):
            aliases = aliases_da_resposta(pergunta["Answer"])
            ranking = pesquisar(pergunta, k=10)
            acertos = [contem_resposta(item["texto"], aliases) for item in ranking]
            registos.append({
                "id": pergunta["QuestionId"],
                "resultados": ranking,
                **{f"acerto@{k}": any(acertos[:k]) for k in (1, 5, 10)},
            })
        guardar_json(destino / f"{nome}_{conjunto}.json", registos)
        rankings[nome] = {r["id"]: r["resultados"] for r in registos}
        print(f"{conjunto} | {nome} | {len(registos)} perguntas")
        for k in (1, 5, 10):
            acertos = sum(r[f"acerto@{k}"] for r in registos)
            print(f"Recall@{k}: {acertos / len(registos):.1%} ({acertos}/{len(registos)})")
    return rankings

rankings_validacao = avaliar_recuperacao(validacao_200, "validacao_200")
rankings_teste = avaliar_recuperacao(teste, "teste_318")


validacao_200: tfidf:   0%|          | 0/200 [00:00<?, ?it/s]

validacao_200 | tfidf | 200 perguntas
Recall@1: 58.0% (116/200)
Recall@5: 83.0% (166/200)
Recall@10: 87.0% (174/200)


validacao_200: bm25:   0%|          | 0/200 [00:00<?, ?it/s]

validacao_200 | bm25 | 200 perguntas
Recall@1: 53.5% (107/200)
Recall@5: 82.5% (165/200)
Recall@10: 88.5% (177/200)


teste_318: tfidf:   0%|          | 0/318 [00:00<?, ?it/s]

teste_318 | tfidf | 318 perguntas
Recall@1: 65.1% (207/318)
Recall@5: 91.8% (292/318)
Recall@10: 96.9% (308/318)


teste_318: bm25:   0%|          | 0/318 [00:00<?, ?it/s]

teste_318 | bm25 | 318 perguntas
Recall@1: 71.7% (228/318)
Recall@5: 92.5% (294/318)
Recall@10: 96.5% (307/318)


## 4. Generator and evaluation

The prompt and decoding settings are shared across conditions. RAG uses five retrieved passages. Oracle uses up to five passages from the question’s associated documents that contain a normalized answer alias, in TF-IDF rank order. Compare Oracle with the other conditions on the common subset where Oracle passages exist.

In [6]:
def gerar_resposta(pergunta, contexto=""):
    mensagens = [
        {
            "role": "system",
            "content": (
                "Answer the question with only a short answer in English. "
                "Use the context when it is helpful. "
                "If the context is empty, answer using your knowledge. "
                "Do not include explanations."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{contexto}\n\nQuestion: {pergunta}\n\nAnswer:"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        mensagens, tokenize=False, add_generation_prompt=True
    )
    entradas = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).to(modelo.device)

    with torch.inference_mode():
        saida = modelo.generate(
            **entradas,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    novos_tokens = saida[0, entradas["input_ids"].shape[1]:]
    return tokenizer.decode(novos_tokens, skip_special_tokens=True).strip()



def formatar_contexto(excertos):
    return "\n\n".join(
        f"[Excerto {i} | {item['ficheiro']}]\n{item['texto']}"
        for i, item in enumerate(excertos, start=1)
    )


In [7]:
REVISOES = {
    "1.5b": ("Qwen/Qwen2.5-1.5B-Instruct", "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"),
    "3b": ("Qwen/Qwen2.5-3B-Instruct", "aa8e72537993ba99e69dfaafa59ed015b17504d1"),
}

def carregar_modelo(tamanho):
    global modelo, tokenizer, MODEL_ID, MODEL_REVISION
    if "modelo" in globals():
        del modelo
    gc.collect()
    torch.cuda.empty_cache()
    MODEL_ID, MODEL_REVISION = REVISOES[tamanho]
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    modelo = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, dtype=torch.float16, device_map={"": 0}
    )
    modelo.eval()
    print(MODEL_ID, MODEL_REVISION, modelo.device)

def avaliar_geracao(perguntas, rankings, conjunto, tamanho):
    if (MODEL_ID, MODEL_REVISION) != REVISOES[tamanho]:
        raise RuntimeError("O gerador carregado não corresponde à avaliação.")
    ficheiro = destino / f"geracao_{conjunto}_qwen{tamanho}.json"
    resultados = {
        r["id"]: r for r in ler_json(ficheiro)
    } if ficheiro.is_file() else {}
    pendentes = [p for p in perguntas if p["QuestionId"] not in resultados]

    for numero, pergunta in enumerate(tqdm(pendentes, desc=f"{conjunto} | Qwen {tamanho}"), 1):
        pid = pergunta["QuestionId"]
        enunciado = pergunta["Question"]
        resposta = pergunta["Answer"]
        aliases = aliases_da_resposta(resposta)
        oracle = []
        for item in pesquisar_excertos(pergunta, k=None):
            if contem_resposta(item["texto"], aliases):
                oracle.append(item)
                if len(oracle) == 5:
                    break

        previsoes = {"closed_book": gerar_resposta(enunciado)}
        for metodo, ranking in rankings.items():
            previsoes[f"rag_{metodo}"] = gerar_resposta(
                enunciado, formatar_contexto(ranking[pid][:5])
            )
        previsoes["oracle"] = (
            gerar_resposta(enunciado, formatar_contexto(oracle)) if oracle else None
        )
        resultados[pid] = {
            "id": pid,
            "pergunta": enunciado,
            "referencia": resposta["Value"],
            "aliases": sorted(aliases),
            "previsoes": previsoes,
            "modelo": MODEL_ID,
            "revisao": MODEL_REVISION,
        }
        if numero % 10 == 0:
            guardar_json(ficheiro, list(resultados.values()))

    guardar_json(ficheiro, list(resultados.values()))
    linhas = [resultados[p["QuestionId"]] for p in perguntas]
    com_oracle = [r for r in linhas if r["previsoes"]["oracle"] is not None]
    metodos = ("closed_book", "rag_tfidf", "rag_bm25")

    for titulo, grupo, condicoes in [
        (conjunto, linhas, metodos),
        ("Subconjunto com Oracle", com_oracle, (*metodos, "oracle")),
    ]:
        print(f"\n{titulo} ({len(grupo)} perguntas)")
        if not grupo:
            continue
        for condicao in condicoes:
            medias = []
            for metrica in (scorer.exact_match_score, scorer.f1_score):
                valores = [
                    max(metrica(r["previsoes"][condicao], a) for a in r["aliases"])
                    for r in grupo
                ]
                medias.append(sum(valores) / len(grupo))
            print(f"{condicao}: EM = {medias[0]:.1%} | F1 = {medias[1]:.1%}")
    return linhas


In [8]:
carregar_modelo("1.5b")
avaliar_geracao(validacao_200, rankings_validacao, "validacao_200", "1.5b")
teste_15b = avaliar_geracao(teste, rankings_teste, "teste_318", "1.5b")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

... (4 outputs left out)

validacao_200 | Qwen 1.5b:   0%|          | 0/200 [00:00<?, ?it/s]


validacao_200 (200 perguntas)
closed_book: EM = 24.5% | F1 = 30.8%
rag_tfidf: EM = 54.5% | F1 = 63.2%
rag_bm25: EM = 55.0% | F1 = 62.6%

Subconjunto com Oracle (196 perguntas)
closed_book: EM = 25.0% | F1 = 31.5%
rag_tfidf: EM = 55.6% | F1 = 64.4%
rag_bm25: EM = 56.1% | F1 = 63.8%
oracle: EM = 61.2% | F1 = 69.7%


teste_318 | Qwen 1.5b:   0%|          | 0/318 [00:00<?, ?it/s]


teste_318 (318 perguntas)
closed_book: EM = 32.4% | F1 = 38.1%
rag_tfidf: EM = 63.5% | F1 = 72.1%
rag_bm25: EM = 69.5% | F1 = 76.2%

Subconjunto com Oracle (317 perguntas)
closed_book: EM = 32.5% | F1 = 38.2%
rag_tfidf: EM = 63.7% | F1 = 72.4%
rag_bm25: EM = 69.7% | F1 = 76.5%
oracle: EM = 77.0% | F1 = 84.3%


In [9]:
carregar_modelo("3b")
teste_3b = avaliar_geracao(teste, rankings_teste, "teste_318", "3b")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2.5-3B-Instruct aa8e72537993ba99e69dfaafa59ed015b17504d1 cuda:0


teste_318 | Qwen 3b:   0%|          | 0/318 [00:00<?, ?it/s]


teste_318 (318 perguntas)
closed_book: EM = 40.9% | F1 = 47.4%
rag_tfidf: EM = 64.5% | F1 = 72.6%
rag_bm25: EM = 65.4% | F1 = 73.7%

Subconjunto com Oracle (317 perguntas)
closed_book: EM = 41.0% | F1 = 47.5%
rag_tfidf: EM = 64.7% | F1 = 72.9%
rag_bm25: EM = 65.6% | F1 = 74.0%
oracle: EM = 73.2% | F1 = 81.6%


## 5. Error analysis

Compare changes in Exact Match and measure how often each generator answers correctly when its five retrieved passages contain an answer alias.

In [10]:
for tamanho, linhas in [("1.5B", teste_15b), ("3B", teste_3b)]:
    print(f"\nQwen {tamanho}")
    for metodo in ("tfidf", "bm25"):
        recuperacao = {
            r["id"]: r["acerto@5"]
            for r in ler_json(destino / f"{metodo}_teste_318.json")
        }
        mudancas = Counter()
        por_alias = Counter()
        exemplos = {}

        for linha in linhas:
            previsoes = linha["previsoes"]
            aliases = linha["aliases"]
            sem = any(scorer.exact_match_score(previsoes["closed_book"], a) for a in aliases)
            com = any(scorer.exact_match_score(previsoes[f"rag_{metodo}"], a) for a in aliases)
            mudancas[sem, com] += 1
            por_alias[recuperacao[linha["id"]], com] += 1
            exemplos.setdefault((sem, com), linha)

        print(f"\n{metodo.upper()}")
        for par, descricao in [
            ((False, True), "Ajudou"), ((True, False), "Prejudicou"),
            ((True, True), "Manteve o acerto"), ((False, False), "Manteve a falha"),
        ]:
            print(f"{descricao}: {mudancas[par]}")
        for presente in (True, False):
            acertos = por_alias[presente, True]
            total = acertos + por_alias[presente, False]
            if total:
                descricao = "Com alias" if presente else "Sem alias"
                print(f"{descricao}: {acertos}/{total} acertos ({acertos / total:.1%})")
        for par in ((False, True), (True, False)):
            if par in exemplos:
                exemplo = exemplos[par]
                print(
                    exemplo["pergunta"],
                    f"Referência: {exemplo['referencia']}",
                    f"Sem documentos: {exemplo['previsoes']['closed_book']}",
                    f"Com RAG: {exemplo['previsoes'][f'rag_{metodo}']}",
                    sep="\n",
                )



Qwen 1.5B

TFIDF
Ajudou: 121
Prejudicou: 22
Manteve o acerto: 81
Manteve a falha: 94
Com alias: 198/292 acertos (67.8%)
Sem alias: 4/26 acertos (15.4%)
In which country is the castle that gives the Habsburg dynasty its name?
Referência: Switzerland
Sem documentos: Austria-Hungary
Com RAG: Switzerland
What is the Milky Way?
Referência: The local galaxy
Sem documentos: our galaxy
Com RAG: barrier to the formation of stars

BM25
Ajudou: 137
Prejudicou: 19
Manteve o acerto: 84
Manteve a falha: 78
Com alias: 216/294 acertos (73.5%)
Sem alias: 5/24 acertos (20.8%)
Who first drew Mickey Mouse when ?Disney first supplied the voice?
Referência: Ub Iwerks
Sem documentos: 1928
Com RAG: Ub Iwerks
What is the Milky Way?
Referência: The local galaxy
Sem documentos: our galaxy
Com RAG: barred spiral galaxy

Qwen 3B

TFIDF
Ajudou: 95
Prejudicou: 20
Manteve o acerto: 110
Manteve a falha: 93
Com alias: 201/292 acertos (68.8%)
Sem alias: 4/26 acertos (15.4%)
Who first drew Mickey Mouse when ?Disney firs

## Results from the original Colab run

These results were copied from the recorded runs. Running the cells above generates a new set of outputs.

| Retriever | Recall@1 | Recall@5 | Recall@10 |
|---|---:|---:|---:|
| TF-IDF | 65.1% | 91.8% | 96.9% |
| BM25 | 71.7% | 92.5% | 96.5% |

| Condition | Questions | Qwen 1.5B EM / F1 | Qwen 3B EM / F1 |
|---|---:|---:|---:|
| Closed-book | 318 | 32.4% / 38.1% | 40.9% / 47.4% |
| RAG with TF-IDF | 318 | 63.5% / 72.1% | 64.5% / 72.6% |
| RAG with BM25 | 318 | 69.5% / 76.2% | 65.4% / 73.7% |
| Closed-book, Oracle subset | 317 | 32.5% / 38.2% | 41.0% / 47.5% |
| RAG with TF-IDF, Oracle subset | 317 | 63.7% / 72.4% | 64.7% / 72.9% |
| RAG with BM25, Oracle subset | 317 | 69.7% / 76.5% | 65.6% / 74.0% |
| Oracle | 317 | 77.0% / 84.3% | 73.2% / 81.6% |

For Qwen 1.5B, BM25 RAG corrected 137 closed-book errors and turned 19 correct answers into errors. When an answer alias appeared in the BM25 context, the model answered exactly in 216/294 cases. Without an alias it answered exactly in 5/24 cases. Alias presence measures lexical coverage and can include passages that merely mention the answer.